# Stage 2 Notebook 39 - Exp2II Anchor-based head + Exp2EE training tricks

**Why this exists.** A core observation across the Exp2 series:

| Architecture | oracle_f1 ceiling | decoded_f1 |
|---|---:|---:|
| Exp2N (CLRKDLaneHead, 192 anchors) | **0.273** | 0.011 (cls broken) |
| Exp2P-FF (LaneQueryHead, 12 queries) | 0.10 | 0.04 (cls works) |

The anchor-based architecture had **2.7x higher geometry ceiling**, but its cls supervision broke (all anchors got the same low score). We pivoted to query head because cls worked, but we **lowered the geometry ceiling**.

Exp2II tries the missing experiment: **CLRKDLaneHead anchor architecture + Exp2EE's training stability fixes**:
- Anchor head (192 priors with ROI gather along each anchor) -- the high-ceiling geometry.
- Cosine LR with linear warmup (Exp2EE's fix for late-training divergence).
- Uncertainty multi-task weighting (Exp2Z's monotonic loss balance).
- Mask auxiliary supervision (Exp2W's proven +65% lift).
- Dynamic-k matching with top-k=4 (CLRKD's published recipe).
- 20 epochs (more time for cls to learn separation over 192 anchors).

Hypothesis: Exp2N's cls broke partly because of training instability (lambda oscillation, no LR decay) on top of the inherent imbalance (5 GTs vs 192 anchors). With modern training tricks, cls might separate enough to use the anchor head's 0.27 geometry ceiling. **If this works, decoded_f1 could jump substantially because the geometry ceiling is far higher.**

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 20-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp34_rmt_gca_anchor_clrkd_modern_training_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp34_rmt_gca_anchor_clrkd_modern_training_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp34_rmt_gca_anchor_clrkd_modern_training_joint_smoke.log
OK exp34_rmt_gca_anchor_clrkd_modern_training_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.2711 det_loss=3.4624 grad_cos=0.4564 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4987602233886719, 'gate/lane_mean': 0.49965766072273254, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp34_rmt_gca_anchor_clrkd_modern_training_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp34_rmt_gca_anchor_clrkd_modern_training_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp34_rmt_gca_anchor_clrkd_modern_training_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp34_rmt_gca_anchor_clrkd_modern_training_joint_short20.tar --epochs 20 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp34_rmt_gca_anchor_clrkd_modern_training_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp34_rmt_gca_anchor_clrkd_modern_training_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp34_rmt_gca_anchor_clrkd_modern_traini

0

## What to watch in Exp2II training

Reference Exp2N (anchor head, no training fixes): oracle_f1=0.273, decoded_f1=0.011 (cls broken).
Reference Exp2EE (query head, all training fixes): oracle_f1=0.107, decoded_f1=0.047.

Pass criteria at epoch 20:
- **`val/lane/decoded_oracle_f1 >= 0.20`**: anchor architecture's high geometry ceiling appears, consistent with Exp2N's 0.27.
- **`val/lane_exist_best_f1 >= 0.65`**: cls separation across 192 anchors is much harder; even moderate progress is meaningful.
- **`val/lane/decoded_f1 >= 0.07`**: substantially beats Exp2EE's 0.047 (1.5x). The anchor-head's higher ceiling translated into cls-ranked outputs.
- **No late-epoch collapse**: matched_iou never drops below 0.10 (cosine LR active).

If decoded_f1 stays below 0.05: anchor head's cls supervision is genuinely intractable for our dataset/model size. Pivot back to query head + larger backbone or external teacher KD.

If oracle_f1 jumps to >= 0.25 but decoded_f1 stays low: cls is the only blocker; KD from a stronger source (external CLRKDNet teacher) is the next move.